In [ ]:
# Lecture 5 — Your First Spark Program: Word Count
# Spark 101 · Module 2 · Lecture 5
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# Build the input DataFrame from an inline passage — no file upload, no Volume, no S3.
lines = [
    "To be, or not to be, that is the question:",
    "Whether tis nobler in the mind to suffer",
    "The slings and arrows of outrageous fortune,",
    "Or to take arms against a sea of troubles",
    "And by opposing end them. To die, to sleep,",
    "No more; and by a sleep to say we end",
    "The heart-ache and the thousand natural shocks",
    "That flesh is heir to.",
]

df = spark.createDataFrame([(line,) for line in lines], ["value"])
df.show(truncate=False)

In [ ]:
# The word count chain — transformations only. Nothing runs yet (lazy).
from pyspark.sql.functions import col, count, explode, lower, split

word_counts = (
    df.select(explode(split(lower(col("value")), r"\W+")).alias("word"))
      .filter(col("word") != "")
      .groupBy("word")
      .agg(count("*").alias("n"))
      .orderBy(col("n").desc())
)

In [ ]:
# Trigger execution — .show() is an action.
word_counts.show(20, truncate=False)

In [ ]:
# Different action, different result.
print(f"Distinct words: {word_counts.count()}")

In [ ]:
# Add a column: the length of each word.
from pyspark.sql.functions import length
word_counts.withColumn("len", length("word")).show(10, truncate=False)

In [ ]:
# Keep only words that appear more than once.
word_counts.filter(word_counts.n > 1).show(truncate=False)

In [ ]:
# The single most common word (first() is an action).
top = word_counts.first()
print("most common:", top["word"], "->", top["n"])